# TFG lechugas · entrenamiento de la etapa 2 en Colab (GPU)

Este notebook **solo orquesta**: comprueba la GPU, monta Drive, deja el repositorio en un commit concreto, instala `requirements-train.txt` **sin tocar torch**, descomprime el dataset de recortes `etapa2_v1` en `/content` y entrena el clasificador de los seis grupos (`python -m src.train.etapa2`). Toda la lógica está en `src/`; los parámetros, en `configs/train/etapa2_effb0.yaml`.

**Antes de empezar**
1. Entorno de ejecución → Cambiar tipo de entorno → GPU T4.
2. Comprime en local la carpeta `data/processed/etapa2_v1` (la que genera `python -m src.data.etapa2`, con `recortes.csv` e `imagenes/`) y súbela a Drive como `MyDrive/tfg-lechugas/data/processed/etapa2_v1.zip`. Los datos no van al repositorio.
3. En la celda de parámetros pon en `COMMIT` el commit exacto y en `EXP_ID` un nombre nuevo para la ejecución (no se pisa una ejecución anterior).
4. Antes del primer entrenamiento completo, ejecuta el modo `smoke` (celda g).

**Qué produce**
- `/content/TFG-VisionArtificial/data/processed/etapa2_v1`: el dataset descomprimido (se lee del disco de Colab, no de Drive). **No se copia a Drive.**
- `/content/TFG-VisionArtificial/runs/<EXP_ID>`: la ejecución (configuración, entorno, datos, métricas por época, resumen con valid y test, matrices de confusión y pesos `mejor.pt` y `ultimo.pt`). Se copia a `MyDrive/tfg-lechugas/runs/<EXP_ID>`.

La primera vez, timm descarga los pesos de EfficientNet-B0 preentrenada en ImageNet (`timm/efficientnet_b0.ra_in1k`, Apache-2.0) desde Hugging Face.

In [ ]:
# a) Parámetros. Es lo único que se edita antes de ejecutar (y MODO, en la celda g).
import os
import shlex
import subprocess
import sys

REPO_URL = "https://github.com/diegocalvofm-ux/TFG-VisionArtificial.git"
COMMIT = "main"  # rama, etiqueta o hash de commit; para reproducir un entrenamiento, un hash
EXP_ID = "etapa2_effb0_001"  # carpeta runs/<EXP_ID>; cámbialo en cada ejecución
CARPETA_DRIVE = "/content/drive/MyDrive/tfg-lechugas"
ZIP_DATOS = f"{CARPETA_DRIVE}/data/processed/etapa2_v1.zip"  # recortes de la v2-01, fuera del repositorio
CONFIG = "configs/train/etapa2_effb0.yaml"

# Derivados: no hace falta tocarlos. Las rutas no deben llevar espacios.
REPO = "/content/TFG-VisionArtificial"
RUNS = f"{REPO}/runs"
DATASET = f"{REPO}/data/processed/etapa2_v1"  # donde lo espera datos.dataset del YAML (en /content)
PYTHON = sys.executable


def ejecutar(comando, cwd=None, comprobar=True):
    """Lanza un comando, enseña su salida según va saliendo y devuelve su código de salida.

    En lenguaje llano: es como escribir el comando en una terminal. Si el comando falla
    (código distinto de 0) lanza un error y el notebook se para en esa celda, en vez de seguir
    con un entorno a medias. Con comprobar=False solo devuelve el código.
    """
    print(f"$ {comando}", flush=True)
    entorno = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    proceso = subprocess.Popen(
        shlex.split(comando),
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=entorno,
    )
    for linea in proceso.stdout:
        print(linea, end="", flush=True)
    codigo = proceso.wait()
    if comprobar and codigo != 0:
        raise RuntimeError(f"El comando terminó con código {codigo}: {comando}")
    return codigo

In [ ]:
# b) GPU. Para aquí si el entorno no tiene GPU (sería un entrenamiento de horas en CPU).
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU: Entorno de ejecución -> Cambiar tipo de entorno -> GPU T4")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# c) Drive. Monta tu Drive y comprueba que el ZIP de los recortes está donde se espera.
import os

from google.colab import drive

drive.mount("/content/drive")

if not os.path.isfile(ZIP_DATOS):
    raise FileNotFoundError(
        f"No existe {ZIP_DATOS}. Comprime data/processed/etapa2_v1 (con recortes.csv e "
        "imagenes/), súbelo a esa ruta de Drive como etapa2_v1.zip (o corrige ZIP_DATOS en la "
        "celda de parámetros) y vuelve a ejecutar."
    )
print(f"ZIP de los recortes: {ZIP_DATOS} ({os.path.getsize(ZIP_DATOS) / 1024**2:.1f} MB)")

In [ ]:
# d) Repositorio. Lo clona (o lo actualiza) y lo deja en el commit pedido.
import os

if os.path.isdir(os.path.join(REPO, ".git")):
    ejecutar("git fetch --all --tags", cwd=REPO)
else:
    ejecutar(f"git clone {REPO_URL} {REPO}")
ejecutar(f"git checkout {COMMIT}", cwd=REPO)
# Si COMMIT es una rama se pone al día; con un hash o una etiqueta no hay nada que actualizar.
if ejecutar("git symbolic-ref -q HEAD", cwd=REPO, comprobar=False) == 0:
    ejecutar("git pull --ff-only", cwd=REPO)
os.chdir(REPO)
ejecutar("git rev-parse --short HEAD")  # el commit exacto que se va a entrenar: apúntalo

In [ ]:
# e) Dependencias SIN tocar torch. Colab ya trae torch y torchvision: se guardan sus versiones, se
# instala con un constraints que las fija y se comprueba después que no cambiaron (si cambian, el
# notebook se para aquí).
ANTES = "/content/torch_antes.json"
DESPUES = "/content/torch_despues.json"
CONSTRAINTS = "/content/constraints.txt"

ejecutar(f"{PYTHON} -m src.train.entorno versiones --out {ANTES}", cwd=REPO)
ejecutar(f"{PYTHON} -m src.train.entorno constraints --out {CONSTRAINTS}", cwd=REPO)
ejecutar(f"{PYTHON} -m pip install -q -c {CONSTRAINTS} -r requirements.txt -r requirements-train.txt", cwd=REPO)
ejecutar(f"{PYTHON} -m src.train.entorno versiones --out {DESPUES}", cwd=REPO)
ejecutar(f"{PYTHON} -m src.train.entorno comprobar --antes {ANTES} --despues {DESPUES}", cwd=REPO)

In [ ]:
# f) Datos. Descomprime etapa2_v1.zip en data/processed/etapa2_v1 del repositorio clonado (en
# /content, donde lo espera datos.dataset del YAML): leer las imágenes de Drive sería lento. Vale un
# ZIP con recortes.csv en la raíz o con la carpeta etapa2_v1/ entera. Nada de esto va a Drive.
import os
import zipfile

if os.path.isfile(f"{DATASET}/recortes.csv"):
    print(f"{DATASET} ya está descomprimido: no se repite (reinicia el entorno para empezar de cero)")
else:
    with zipfile.ZipFile(ZIP_DATOS) as zf:
        nombres = set(zf.namelist())
        if "recortes.csv" in nombres:
            destino = DATASET  # el ZIP trae el contenido de la carpeta
        elif "etapa2_v1/recortes.csv" in nombres:
            destino = os.path.dirname(DATASET)  # el ZIP trae la carpeta etapa2_v1 entera
        else:
            raise ValueError(
                f"{ZIP_DATOS} no tiene recortes.csv ni etapa2_v1/recortes.csv: comprime la "
                "carpeta data/processed/etapa2_v1 (con recortes.csv e imagenes/)"
            )
        os.makedirs(destino, exist_ok=True)
        zf.extractall(destino)
imagenes = [n for n in os.listdir(f"{DATASET}/imagenes") if n.lower().endswith(".jpg")]
print(f"{len(imagenes)} imágenes en {DATASET}/imagenes")

In [ ]:
# g) Modo. "smoke" = 1 época con unos pocos recortes de train, valid y test (prueba rápida: hazla
# antes del primer entrenamiento completo). "completo" = el entrenamiento entero del YAML.
MODO = "smoke"  # "smoke" | "completo"

In [ ]:
# h) Entrenamiento. Escribe runs/<EXP_ID>/ y, al final, imprime la línea REGISTRO: y el comando
# para anotar el experimento. El dataset descomprimido solo se lee.
if MODO not in ("smoke", "completo"):
    raise ValueError(f'MODO debe ser "smoke" o "completo" (es {MODO!r})')
opcion_smoke = " --smoke" if MODO == "smoke" else ""
ejecutar(
    f"{PYTHON} -m src.train.etapa2 --config {CONFIG} --exp-id {EXP_ID}{opcion_smoke}",
    cwd=REPO,
)

In [ ]:
# i) Copia la ejecución a Drive (para no perderla al cerrarse la sesión de Colab). Solo va
# runs/<EXP_ID>: el dataset descomprimido se queda en /content.
import os
import shutil

ORIGEN = f"{RUNS}/{EXP_ID}"
DESTINO = f"{CARPETA_DRIVE}/runs/{EXP_ID}"
shutil.copytree(ORIGEN, DESTINO, dirs_exist_ok=True)  # repetirla sobrescribe, no falla
for carpeta, _, archivos in sorted(os.walk(DESTINO)):
    for nombre in sorted(archivos):
        ruta = os.path.join(carpeta, nombre)
        kilobytes = os.path.getsize(ruta) / 1024
        print(f"{os.path.relpath(ruta, DESTINO)}  ({kilobytes:.1f} KB)")
print(f"Copiado en {DESTINO}")

## Después

1. Descarga la carpeta `MyDrive/tfg-lechugas/runs/<EXP_ID>` de Drive a `runs/<EXP_ID>` del repositorio local (la carpeta `runs/` no se sube a git).
2. Mira `runs/<EXP_ID>/resumen.json`: mejor época y su criterio (media de la exactitud equilibrada en valid de los grupos con datos), métricas de valid y test con el mejor punto frente al clasificador que siempre dice la clase mayoritaria (global y por fuente), avisos, tiempo de entrenamiento, memoria de la GPU y pesos en `pesos/`. Las matrices de confusión están en `matrices_confusion.json`.
3. Lee los avisos: con los datos actuales la fuente propia solo está en train, así que valid y test solo miden `daño_bio`.
4. Si no era un `smoke`, anota el experimento con Claude Code. La salida de la celda h trae la línea que empieza por `REGISTRO:` (commit, GPU y versiones) y el comando `python -m src.experimentos.registro campos --exp expNNN --set ...` con los campos de la cabecera ya rellenos; pega la línea `REGISTRO:` en las observaciones y usa `/registrar-experimento`. `resumen.json` guarda los mismos campos. No se inventa ninguna métrica: si el entrenamiento falló, el valor queda vacío y el error está en `resumen.json`.